
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/08_error_handling/08_error_handling.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 08 — Error Handling & Debugging

**Learning Objectives:** try/except, exception types, custom exceptions, logging, debugging techniques

**Estimated time:** 45–60 minutes

---

## 8.1 Why Errors Are Normal (and Necessary)

**The mindset shift:**
Beginners treat errors as failures. Experienced programmers treat them as information. An unhandled error tells you exactly what went wrong, where, and why — that is more useful than silent wrong results.

**Two types of errors in Python:**

**Syntax errors** happen before your code runs. Python cannot even parse the code:
```python
if x = 5:   # SyntaxError: should be ==
```

**Runtime errors (exceptions)** happen while the code is running. The code was valid Python, but something went wrong during execution — a file was missing, a number was divided by zero, a key did not exist in a dictionary.

**What happens without error handling:**
When an unhandled exception occurs, Python immediately stops the program and prints a **traceback** — a stack of all the function calls that led to the crash. Learning to read tracebacks is one of the most valuable debugging skills you can develop.

**What error handling lets you do:**
- Catch specific errors and respond gracefully instead of crashing
- Show useful error messages to users instead of technical tracebacks
- Try to recover from the error and continue running
- Log the error for later debugging without interrupting the user

## 8.2 The try / except Block

**How it works:**
Python first tries to execute the code inside `try`. If an exception occurs, it immediately jumps to the matching `except` block. If no exception occurs, the `except` block is skipped entirely.

**The four clauses:**
- `try` — the code that might raise an exception
- `except ExceptionType` — runs only if that specific exception was raised
- `else` — runs only if NO exception was raised in `try`
- `finally` — runs ALWAYS, whether an exception occurred or not

**Best practice — catch specific exceptions:**
Always name the specific exception you expect (`except ValueError`) rather than using a bare `except:` which catches everything including keyboard interrupts and system exits. Catching too broadly hides bugs.

In [ ]:
# Without error handling — this crashes the whole program
# result = 10 / 0   # ZeroDivisionError: division by zero

# With error handling — we control what happens
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
    result = None

print(f"Result: {result}")

print()

# Catching the exception object gives you the error message
try:
    value = int("hello")
except ValueError as e:
    print(f"Conversion failed: {e}")
    value = 0

print(f"Value: {value}")

In [ ]:
# Multiple except blocks — handle different errors differently
def safe_divide(a, b):
    try:
        result = a / b
        
    except ZeroDivisionError:
        print(f"  Error: cannot divide {a} by zero")
        return None
    
    except TypeError as e:
        print(f"  Error: wrong types given — {e}")
        return None
    
    else:
        # Runs ONLY if no exception occurred
        print(f"  Success: {a} / {b} = {result:.4f}")
        return result
    
    finally:
        # Runs ALWAYS — used for cleanup (closing files, db connections, etc.)
        print(f"  [Operation attempted: {a} / {b}]")

safe_divide(10, 3)
print()
safe_divide(10, 0)
print()
safe_divide("ten", 2)

**When to use `finally`:**
The `finally` block is designed for cleanup code that MUST run regardless of whether an error occurred. Common examples: closing a file, releasing a database connection, deleting a temporary file. In modern Python you usually use `with` statements for this instead, but `finally` is still useful.

## 8.3 Common Built-in Exceptions

**Python's exception hierarchy:**
All exceptions inherit from `BaseException`. The ones you will encounter most are subclasses of `Exception`. Understanding which exception to expect in which situation lets you write precise, targeted error handling.

**Memory aid:** The exception name usually tells you exactly what went wrong. `ValueError` = right type, wrong value. `TypeError` = wrong type entirely. `KeyError` = key missing. `IndexError` = index out of range.

In [ ]:
# The most common exceptions with real examples

# ValueError — right type, wrong value
try:
    age = int("twenty")
except ValueError as e:
    print(f"ValueError: {e}")

# TypeError — wrong type for the operation
try:
    result = "hello" + 5
except TypeError as e:
    print(f"TypeError: {e}")

# KeyError — dictionary key does not exist
try:
    d = {"name": "Alice"}
    print(d["age"])
except KeyError as e:
    print(f"KeyError: {e}")

# IndexError — list index out of range
try:
    lst = [1, 2, 3]
    print(lst[10])
except IndexError as e:
    print(f"IndexError: {e}")

# AttributeError — object does not have that attribute/method
try:
    x = 42
    x.upper()
except AttributeError as e:
    print(f"AttributeError: {e}")

# FileNotFoundError — file does not exist
try:
    with open("nonexistent_file.txt") as f:
        content = f.read()
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

# ZeroDivisionError — division or modulo by zero
try:
    result = 5 % 0
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

## 8.4 Custom Exceptions

**Why create custom exceptions?**
Built-in exceptions are generic. When you raise a `ValueError`, anyone reading the code has to guess the context. A custom exception like `InsufficientFundsError` or `InvalidAgeError` is self-documenting — its name instantly communicates what went wrong and why.

**How to create custom exceptions:**
Simply subclass `Exception` (or any more specific built-in exception). You can keep them empty (just `pass`) for basic use, or add custom attributes to carry extra context about the error.

**When to use them:**
In any application or library where you want callers to be able to catch your specific errors separately from all other errors. This is standard practice in professional Python code.

In [ ]:
# Basic custom exception — just a named class
class ValidationError(Exception):
    """Raised when input data fails validation rules."""
    pass

# Custom exception with extra information
class InsufficientFundsError(Exception):
    """Raised when a withdrawal exceeds the account balance."""
    def __init__(self, requested, available):
        self.requested  = requested
        self.available  = available
        self.shortfall  = requested - available
        # Always call super().__init__() with the human-readable message
        super().__init__(
            f"Requested £{requested:,.2f} but only £{available:,.2f} available "
            f"(shortfall: £{self.shortfall:,.2f})"
        )

# Using custom exceptions
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner    = owner
        self._balance = balance
    
    def withdraw(self, amount):
        if amount <= 0:
            raise ValidationError("Withdrawal amount must be positive")
        if amount > self._balance:
            raise InsufficientFundsError(amount, self._balance)
        self._balance -= amount
        return self._balance

acc = BankAccount("Alice", 500)

# Caller can catch specific errors and respond differently to each
for amount in [200, -50, 1000]:
    try:
        new_balance = acc.withdraw(amount)
        print(f"  Withdrew £{amount}. Balance: £{new_balance}")
    except InsufficientFundsError as e:
        print(f"  Funds error: {e}")
        print(f"  Shortfall was: £{e.shortfall}")   # custom attribute!
    except ValidationError as e:
        print(f"  Validation error: {e}")

## 8.5 Logging — The Professional Alternative to print()

**Why logging instead of print()?**
`print()` is fine for quick debugging, but professional applications use the `logging` module because:
- You can set a **level** (DEBUG, INFO, WARNING, ERROR, CRITICAL) and filter out less important messages
- Log messages include **timestamps** and **source location** automatically
- You can easily switch from printing to the console to writing to a file, without changing your code
- You can turn off all debug messages in production with one line
- Multiple modules can all log to the same place consistently

**Log levels (in increasing severity):**
DEBUG → INFO → WARNING → ERROR → CRITICAL

In [ ]:
import logging

# Configure logging — do this once at the start of your program
logging.basicConfig(
    level   = logging.DEBUG,
    format  = "%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt = "%H:%M:%S"
)

logger = logging.getLogger(__name__)

def process_records(records):
    logger.info(f"Starting to process {len(records)} records")
    results = []
    errors  = 0
    
    for i, record in enumerate(records):
        try:
            # Simulate processing — convert to int and square it
            value = int(record) ** 2
            results.append(value)
            logger.debug(f"Record {i}: '{record}' -> {value}")
            
        except ValueError:
            errors += 1
            logger.warning(f"Record {i}: could not convert '{record}' to int — skipping")
        
        except Exception as e:
            errors += 1
            logger.error(f"Record {i}: unexpected error — {e}")
    
    if errors > 0:
        logger.warning(f"Completed with {errors} errors out of {len(records)} records")
    else:
        logger.info(f"All {len(records)} records processed successfully")
    
    return results

data = ["4", "9", "hello", "16", "world", "25"]
results = process_records(data)
print("Results:", results)

---

## Key Takeaways

- Use `try/except` to handle exceptions gracefully instead of crashing
- Always catch **specific** exception types — avoid bare `except:`
- `else` runs when no exception occurred; `finally` runs always (used for cleanup)
- Read tracebacks carefully — they tell you exactly where and why the error happened
- Create **custom exceptions** to make error context self-documenting
- Use **logging** instead of `print()` for anything beyond quick debugging

## Next: [09 — Advanced Python](../09_advanced_python/09_advanced_python.ipynb)
